In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
#from torchsummary import summary
from torchinfo import summary

input =  [ [1, 2], [3, 3], [2, 6], [6, 1], [8, 8], [-2, -1], [-8, -2], [-3, -4], [-4, -8], [-6, -5], [-2, 2], [-3, 6], [-4, 4], [-6, 3], [-7, 5], [1, -7], [2, -3], [4, -1], [5, -4], [8, -2] ]
labels = [    [1],    [1],    [1],    [1],    [1],      [1],      [1],      [1],      [1],      [1],    [-1],    [-1],    [-1],    [-1],    [-1],    [-1],    [-1],    [-1],    [-1],    [-1] ]

tensorInput = torch.Tensor(input)
tensorLabels = torch.Tensor(labels)

dataset = TensorDataset(tensorInput, tensorLabels)

In [10]:
# Define the model
class MyModel(nn.Module):
  def __init__(self):
    super(MyModel, self).__init__()
    self.fc1 = nn.Linear(2, 4)  # fully connected layer 1: input layer to hidden layer
    self.fc2 = nn.Linear(4, 1)  # fully connected layer 2:  hidden layer to output layer
    self.tanh = nn.Tanh()       # tanh activation function
    
  def forward(self, x):
    x = self.fc1(x)
    x = self.tanh(x)
    x = self.fc2(x)
    x = self.tanh(x)
    return x
  
  def fwdDbg(self, x):
    x = self.fc1(x)
    xHidden = self.tanh(x)
    yHat = self.fc2(xHidden)
    yHat = self.tanh(yHat)
    return xHidden, yHat

In [11]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device", device)

# model training
torch.manual_seed(42)
model = MyModel().to(device)
summary(model, input_size=(1,2), device=device)

model.train()  # grundsätzlich empfohlen! (siehe auch unten)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

num_epochs = 1000
dataloader = DataLoader(dataset, batch_size=1)

for epoch in range(num_epochs):
  for batch_idx, (x, y) in enumerate(dataloader):
    x, y = x.to(device), y.to(device) # verschiebt Daten auf MPS
    output = model(x)                 # make prediction with model
    loss = criterion(output, y)       # calculate loss from y, y_hat/output

    optimizer.zero_grad() # zero out gradients
    loss.backward()       # backpropagation
    optimizer.step()      # make optimizisation step for training
      
  if epoch % 100 == 0:
    print(f"Epoch {epoch} Loss: {loss:.6f}")
  
  model.eval()  # grundsätzlich empfohlen! (siehe auch oben)

Using device mps
Epoch 0 Loss: 2.031728
Epoch 100 Loss: 0.000205
Epoch 200 Loss: 0.000073
Epoch 300 Loss: 0.000135
Epoch 400 Loss: 0.000099
Epoch 500 Loss: 0.000074
Epoch 600 Loss: 0.000059
Epoch 700 Loss: 0.000048
Epoch 800 Loss: 0.000041
Epoch 900 Loss: 0.000035


In [12]:
# outputs / loss
for batch_idx, (x, y) in enumerate(dataloader):
  x, y = x.to(device), y.to(device) # verschiebt Daten auf MPS
  output = model(x)
  loss = criterion(output, y)

  x_cpu = x.cpu().detach().numpy() # verschiebt Daten zurück auf CPU für numpy
  output_cpu = output.cpu().detach().numpy()
  loss_val = loss.item()

  print( f"{x_cpu[0][0]:.0f} {x_cpu[0][1]:.0f} {output_cpu[0][0]:.4f} {loss_val:.4f}" )  # x, y, out, loss

1 2 0.9902 0.0001
3 3 0.9887 0.0001
2 6 0.9366 0.0040
6 1 0.8995 0.0101
8 8 0.9869 0.0002
-2 -1 0.9867 0.0002
-8 -2 0.9567 0.0019
-3 -4 0.9809 0.0004
-4 -8 0.9580 0.0018
-6 -5 0.9571 0.0018
-2 2 -0.9242 0.0057
-3 6 -0.9887 0.0001
-4 4 -0.9864 0.0002
-6 3 -0.9519 0.0023
-7 5 -0.9686 0.0010
1 -7 -0.9496 0.0025
2 -3 -0.9913 0.0001
4 -1 -0.9143 0.0073
5 -4 -0.9943 0.0000
8 -2 -0.9945 0.0000


In [13]:
# print parameters
for name, param in model.named_parameters():
    if param.requires_grad:
        print (name, param.data)

fc1.weight tensor([[ 0.3719,  0.2844],
        [-0.9518,  0.5787],
        [-0.5995,  1.2312],
        [-0.5748,  0.2728]], device='mps:0')
fc1.bias tensor([ 2.2219, -2.2579,  3.0896,  0.2843], device='mps:0')
fc2.weight tensor([[-2.4155, -2.7423,  2.7252,  0.1866]], device='mps:0')
fc2.bias tensor([-0.3600], device='mps:0')


In [14]:
# all results
for xTest in range(-8, 9):
  for yTest in range(-8, 9):
    #xHidden, yHat = model.fwdDbg( torch.Tensor([xTest, yTest]) )
    with torch.no_grad():
      x_in = torch.tensor([xTest, yTest], dtype=torch.float32).to(device)
      xHidden, yHat = model.fwdDbg(x_in)

    xHidden_cpu = xHidden.cpu().numpy()
    yHat_cpu = yHat.cpu().numpy()
    print( f"{xTest}  {yTest}  {xHidden_cpu[0]:.4f}  {xHidden[1]:.4f}  {xHidden_cpu[2]:.4f}  {xHidden_cpu[3]:.4f}  {yHat_cpu[0]:.4f}" )

-8  -8  -0.9953  0.6211  -0.9614  0.9910  -0.9701
-8  -7  -0.9918  0.8631  -0.6248  0.9948  -0.9516
-8  -6  -0.9855  0.9549  0.4609  0.9970  0.6879
-8  -5  -0.9745  0.9856  0.9390  0.9982  0.9665
-8  -4  -0.9555  0.9955  0.9947  0.9990  0.9713
-8  -3  -0.9227  0.9986  0.9995  0.9994  0.9668
-8  -2  -0.8673  0.9995  1.0000  0.9997  0.9567
-8  -1  -0.7770  0.9999  1.0000  0.9998  0.9337
-8  0  -0.6372  1.0000  1.0000  0.9999  0.8738
-8  1  -0.4375  1.0000  1.0000  0.9999  0.6995
-8  2  -0.1826  1.0000  1.0000  1.0000  0.2456
-8  3  0.0993  1.0000  1.0000  1.0000  -0.4056
-8  4  0.3662  1.0000  1.0000  1.0000  -0.7913
-8  5  0.5839  1.0000  1.0000  1.0000  -0.9218
-8  6  0.7410  1.0000  1.0000  1.0000  -0.9626
-8  7  0.8446  1.0000  1.0000  1.0000  -0.9772
-8  8  0.9090  1.0000  1.0000  1.0000  -0.9832
-7  -8  -0.9902  -0.2213  -0.9882  0.9719  0.1264
-7  -7  -0.9827  0.3397  -0.8698  0.9836  -0.8021
-7  -6  -0.9697  0.7317  -0.1007  0.9905  -0.1134
-7  -5  -0.9472  0.9072  0.8110  0.9945